# Tool use with Claude

## The text edit tool

In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from anthropic.types import Message
import os
import shutil
from typing import Optional, List
import json

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [12]:
def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system = None, temperature = 1.0, stop_sequences = [], tools = None, max_tokens = 4096):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [4]:
class TextEditorTool:
#    def __init__(self, base_dir: str = "", backup_dir: str = ""):
#        self.base_dir = base_dir or os.getcwd()
#        self.backup_dir = backup_dir or os.path.join(self.base_dir, ".backups")
#        os.makedirs(self.backup_dir, exist_ok=True)
    def __init__(
        self,
        base_dir: str = "",
        backup_dir: str = "",
        allowed_dirs: Optional[List[str]] = None,
    ):
        self.base_dir = os.path.realpath(base_dir or os.getcwd())

        self.allowed_dirs = [self.base_dir]
        for d in allowed_dirs or []:
            resolved = os.path.realpath(os.path.join(self.base_dir, d))
            if resolved not in self.allowed_dirs:
                self.allowed_dirs.append(resolved)

        self.backup_dir = os.path.realpath(
            backup_dir or os.path.join(self.base_dir, ".backups")
        )
        os.makedirs(self.backup_dir, exist_ok=True)

#    def _validate_path(self, file_path: str) -> str:
#        abs_path = os.path.normpath(os.path.join(self.base_dir, file_path))
#        if not abs_path.startswith(self.base_dir):
#            raise ValueError(
#                f"Access denied: Path '{file_path}' is outside the allowed directory"
#            )
#        return abs_path
    def _validate_path(self, file_path: str) -> str:
        abs_path = os.path.realpath(os.path.join(self.base_dir, file_path))

        for root in self.allowed_dirs:
            if abs_path == root or abs_path.startswith(root + os.sep):
                return abs_path

        raise ValueError(
            f"Access denied: Path '{file_path}' is outside the allowed directories"
        )

    def _backup_file(self, file_path: str) -> str:
        if not os.path.exists(file_path):
            return ""
        file_name = os.path.basename(file_path)
        backup_path = os.path.join(
            self.backup_dir, f"{file_name}.{os.path.getmtime(file_path):.0f}"
        )
        shutil.copy2(file_path, backup_path)
        return backup_path

    def _restore_backup(self, file_path: str) -> str:
        file_name = os.path.basename(file_path)
        backups = [
            f for f in os.listdir(self.backup_dir) if f.startswith(file_name + ".")
        ]
        if not backups:
            raise FileNotFoundError(f"No backups found for {file_path}")

        latest_backup = sorted(backups, reverse=True)[0]
        backup_path = os.path.join(self.backup_dir, latest_backup)

        shutil.copy2(backup_path, file_path)
        return f"Successfully restored {file_path} from backup"

    def _count_matches(self, content: str, old_str: str) -> int:
        return content.count(old_str)

    def view(self, file_path: str, view_range: Optional[List[int]] = None) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if os.path.isdir(abs_path):
                try:
                    return "\n".join(os.listdir(abs_path))
                except PermissionError:
                    raise PermissionError(
                        "Permission denied. Cannot list directory contents."
                    )

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            if view_range:
                start, end = view_range
                lines = content.split("\n")

                if end == -1:
                    end = len(lines)

                selected_lines = lines[start - 1 : end]

                result = []
                for i, line in enumerate(selected_lines, start):
                    result.append(f"{i}: {line}")

                return "\n".join(result)
            else:
                lines = content.split("\n")
                result = []
                for i, line in enumerate(lines, 1):
                    result.append(f"{i}: {line}")

                return "\n".join(result)

        except UnicodeDecodeError:
            raise UnicodeDecodeError(
                "utf-8",
                b"",
                0,
                1,
                "File contains non-text content and cannot be displayed.",
            )
        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot access file.")
        except Exception as e:
            raise type(e)(str(e))

    def str_replace(self, file_path: str, old_str: str, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            match_count = self._count_matches(content, old_str)

            if match_count == 0:
                raise ValueError(
                    "No match found for replacement. Please check your text and try again."
                )
            elif match_count > 1:
                raise ValueError(
                    f"Found {match_count} matches for replacement text. Please provide more context to make a unique match."
                )

            # Create backup before modifying
            self._backup_file(abs_path)

            # Perform the replacement
            new_content = content.replace(old_str, new_str)

            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(new_content)

            return "Successfully replaced text at exactly one location."

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def create(self, file_path: str, file_text: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            # Check if file already exists
            if os.path.exists(abs_path):
                raise FileExistsError(
                    "File already exists. Use str_replace to modify it."
                )

            # Create parent directories if they don't exist
            os.makedirs(os.path.dirname(abs_path), exist_ok=True)

            # Create the file
            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(file_text)

            return f"Successfully created {file_path}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot create file.")
        except Exception as e:
            raise type(e)(str(e))

    def insert(self, file_path: str, insert_line: int, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            # Create backup before modifying
            self._backup_file(abs_path)

            with open(abs_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            # Handle line endings
            if lines and not lines[-1].endswith("\n"):
                new_str = "\n" + new_str

            # Insert at the beginning if insert_line is 0
            if insert_line == 0:
                lines.insert(0, new_str + "\n")
            # Insert after the specified line
            elif insert_line > 0 and insert_line <= len(lines):
                lines.insert(insert_line, new_str + "\n")
            else:
                raise IndexError(
                    f"Line number {insert_line} is out of range. File has {len(lines)} lines."
                )

            with open(abs_path, "w", encoding="utf-8") as f:
                f.writelines(lines)

            return f"Successfully inserted text after line {insert_line}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def undo_edit(self, file_path: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            return self._restore_backup(abs_path)

        except ValueError as e:
            raise ValueError(str(e))
        except FileNotFoundError:
            raise FileNotFoundError("No previous edits to undo")
        except PermissionError:
            raise PermissionError("Permission denied. Cannot restore file.")
        except Exception as e:
            raise type(e)(str(e))

In [5]:
text_editor_tool = TextEditorTool(allowed_dirs = ["../../scripts"])


def run_tool(tool_name, tool_input):
    if tool_name == "str_replace_based_edit_tool": # Esto se tuvo que ajustar de 'str_replace_editor' a 'str_replace_based_edit_tool'
        command = tool_input["command"]
        if command == "view":
            return text_editor_tool.view(
                tool_input["path"], tool_input.get("view_range")
            )
        elif command == "str_replace":
            return text_editor_tool.str_replace(
                tool_input["path"], tool_input["old_str"], tool_input["new_str"]
            )
        elif command == "create":
            return text_editor_tool.create(tool_input["path"], tool_input["file_text"])
        elif command == "insert":
            return text_editor_tool.insert(
                tool_input["path"],
                tool_input["insert_line"],
                tool_input["new_str"],
            )
        elif command == "undo_edit":
            return text_editor_tool.undo_edit(tool_input["path"])
        else:
            raise Exception(f"Unknown text editor command: {command}")
    else:
        raise Exception(f"Unknown tool name: {tool_name}")


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [6]:
def get_text_edit_schema(model):
    return {
        "type": "text_editor_20250728",
        "name": "str_replace_based_edit_tool",
    }

In [13]:
def run_conversation(messages):
    while True:
        response = chat(
            messages,
            tools=[get_text_edit_schema(model)],
        )

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason == "max_tokens":
            raise Exception("Maximum tokens number reached. Update the limit.")

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [ ]:
messages = []

add_user_message(
    messages,
    """
    Open the ././scripts/main.py file and summarize its contents
    """,
)

run_conversation(messages)

I'll open and read the file at `././scripts/main.py` to summarize its contents for you.

Let me try again with the correct tool name:



I apologize for the confusion. Let me try viewing the file with the correct approach:
I apologize for the technical difficulty. It appears there's an issue with the tool invocation. Let me try once more:














KeyboardInterrupt: 

In [9]:
messages = []

add_user_message(
    messages,
    """
    Open the main.py file and summarize its contents. It is located on a folder called "scripts" who exists two levels above of where we are located, something like this: building-with-the-claude-api/scripts/main.py (we would be here: building-with-the-claude-api/jupyter_notebooks/tool_use_with_claude/)
    """,
)

run_conversation(messages)

I'll help you open and summarize the main.py file. Let me navigate to the correct path.

Let me try viewing the file:












































KeyboardInterrupt: 

In [10]:
messages

[{'role': 'user',
  'content': '\n    Open the main.py file and summarize its contents. It is located on a folder called "scripts" who exists two levels above of where we are located, something like this: building-with-the-claude-api/scripts/main.py (we would be here: building-with-the-claude-api/jupyter_notebooks/tool_use_with_claude/)\n    '},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you open and summarize the main.py file. Let me navigate to the correct path.", type='text'),
   ToolUseBlock(id='toolu_014JXQY6SWggSWk83J7p3cWd', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': '../../scripts/main.py'}, name='str_replace_based_edit_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_014JXQY6SWggSWk83J7p3cWd',
    'content': 'Error: Unknown tool name: str_replace_based_edit_tool',
    'is_error': True}]},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_012nSrFP

> Resulta que el notebook de ejemplo cambió el atributo ```"name"``` de la función ```get_text_edit_schema()``` (de ```str_replace_editor``` a ```str_replace_based_edit_tool```) pero no lo cambió en la validación de la función ```run_tool()```, por lo que sin importar cuántas veces Claude quisiera intentar nuevas formas de hacer la lectura, siempre obtenía la respuesta ```Error: Unknown tool name: str_replace_based_edit_tool``` y entraba en un loop que consumía los tokens infinitamente

In [13]:
messages = []

add_user_message(
    messages,
    """
    Open the ../../scripts/main.py file and summarize its contents
    """,
)

run_conversation(messages)

I'll open and read the file for you.
I apologize, but I'm unable to access the path `../../scripts/main.py` as it's outside my allowed directory. I can only access files within my current working directory.

Could you please either:
1. Provide the absolute path to the file
2. Copy the file contents directly, or
3. Navigate to the correct directory first

If you can tell me what the current working directory should be or provide the correct path relative to an accessible location, I'd be happy to help summarize the file's contents.


[{'role': 'user',
  'content': '\n    Open the ../../scripts/main.py file and summarize its contents\n    '},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll open and read the file for you.", type='text'),
   ToolUseBlock(id='toolu_01HveMZ13Gg9WmYDiuoZTJUu', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': '../../scripts'}, name='str_replace_based_edit_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01HveMZ13Gg9WmYDiuoZTJUu',
    'content': "Error: Access denied: Path '../../scripts' is outside the allowed directory",
    'is_error': True}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I apologize, but I'm unable to access the path `../../scripts/main.py` as it's outside my allowed directory. I can only access files within my current working directory.\n\nCould you please either:\n1. Provide the absolute path to the file\n2. Copy the file contents directl

In [14]:
messages = []

add_user_message(
    messages,
    """
    Open the main.py file and summarize its contents. It is located on a folder called "scripts" who exists two levels above of where we are located, something like this: building-with-the-claude-api/scripts/main.py (we would be here: building-with-the-claude-api/jupyter_notebooks/tool_use_with_claude/)
    """,
)

run_conversation(messages)

I'll help you open and summarize the main.py file. Let me navigate to the correct path.
Let me try with an absolute path approach:
Let me check the current directory structure first:
Let me navigate up to see the parent directory structure:
I apologize, but I'm unable to access the main.py file at the path you specified (../../scripts/main.py). It appears that I have restricted access limited to the current directory only. I'm currently in what seems to be the `tool_use_with_claude` directory based on the files I can see, but I cannot navigate to parent directories or the scripts folder that would be two levels above.

The files I can see in the current directory are various Jupyter notebooks related to tools and streaming:
- Several tool examples (001_tools, 002_tools, 003_tools, etc.)
- Tool streaming examples
- Text editor tool examples

If you need me to access the main.py file, you might need to:
1. Copy the file to the current directory, or
2. Provide the contents directly, or
3.

[{'role': 'user',
  'content': '\n    Open the main.py file and summarize its contents. It is located on a folder called "scripts" who exists two levels above of where we are located, something like this: building-with-the-claude-api/scripts/main.py (we would be here: building-with-the-claude-api/jupyter_notebooks/tool_use_with_claude/)\n    '},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you open and summarize the main.py file. Let me navigate to the correct path.", type='text'),
   ToolUseBlock(id='toolu_01NxzfdYurVKuuhW8W959SQo', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': '../../scripts/main.py'}, name='str_replace_based_edit_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01NxzfdYurVKuuhW8W959SQo',
    'content': "Error: Access denied: Path '../../scripts/main.py' is outside the allowed directory",
    'is_error': True}]},
 {'role': 'assistant',
  'content': [T

> Aquí se tuvieron que refactorizar las funciones ```__init__()``` y ```_validate_path()``` para aceptar la ruta de la carpeta de scripts y además pasarla como atributo ```allowed_dirs=["../../scripts"]``` al instanciar el editor de texto

In [8]:
messages = []

add_user_message(
    messages,
    """
    Open the ../../scripts/main.py file and summarize its contents
    """,
)

run_conversation(messages)

I'll open and summarize the contents of the main.py file for you.

## Summary of ../../scripts/main.py

This is a simple Python script containing:

- **Function**: `greeting()`
  - Prints the message "Hello there!" to the console
  - Contains a comment referencing "General Kenobi!" (a reference to the Star Wars prequel memes)

The file is very minimal with just 2 lines - a single function definition that outputs a greeting when called. The comment appears to be a playful reference to the famous Star Wars scene where Obi-Wan Kenobi responds to General Grievous saying "Hello there!"


[{'role': 'user',
  'content': '\n    Open the ../../scripts/main.py file and summarize its contents\n    '},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll open and summarize the contents of the main.py file for you.", type='text'),
   ToolUseBlock(id='toolu_01GA2jFdLA9P2Hp5A44MqkfE', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': '../../scripts'}, name='str_replace_based_edit_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01GA2jFdLA9P2Hp5A44MqkfE',
    'content': '"main.py"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01Hvy3dD6yAFRgcuTFRmh649', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': '../../scripts/main.py'}, name='str_replace_based_edit_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01Hvy3dD6yAFRgcuTFRmh649',
    'content': '"1: def greetin

In [10]:
messages = []

add_user_message(
    messages,
    """
    Create a new file called 'pi_calculation.py' on the ../../scripts/ directory and write a function in it to calculate pi to the 5th digit.

    Then, create a 'pi_calculation_test.py' file on the same directory to test your implementation.
    """,
)

run_conversation(messages)

I'll help you create both files. Let me start by creating the `pi_calculation.py` file with a function to calculate pi to the 5th digit, and then create a test file for it.


[{'role': 'user',
  'content': "\n    Create a new file called 'pi_calculation.py' on the ../../scripts/ directory and write a function in it to calculate pi to the 5th digit.\n\n    Then, create a 'pi_calculation_test.py' file on the same directory to test your implementation.\n    "},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you create both files. Let me start by creating the `pi_calculation.py` file with a function to calculate pi to the 5th digit, and then create a test file for it.", type='text'),
   ToolUseBlock(id='toolu_016HhXgfuTXLtTo71edph9QH', caller=DirectCaller(type='direct'), input={'command': 'create', 'path': '../../scripts/pi_calculation.py'}, name='str_replace_based_edit_tool', type='tool_use')]}]

In [11]:
messages

[{'role': 'user',
  'content': "\n    Create a new file called 'pi_calculation.py' on the ../../scripts/ directory and write a function in it to calculate pi to the 5th digit.\n\n    Then, create a 'pi_calculation_test.py' file on the same directory to test your implementation.\n    "},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you create both files. Let me start by creating the `pi_calculation.py` file with a function to calculate pi to the 5th digit, and then create a test file for it.", type='text'),
   ToolUseBlock(id='toolu_016HhXgfuTXLtTo71edph9QH', caller=DirectCaller(type='direct'), input={'command': 'create', 'path': '../../scripts/pi_calculation.py'}, name='str_replace_based_edit_tool', type='tool_use')]}]

> Aquí hubo que crear una variable de entrada en la cunfión ```chat()``` y subir el atributo ```max_tokens``` a 4096 para que se pudiera crear el archivo

In [14]:
messages = []

add_user_message(
    messages,
    """
    Create a new file called 'pi_calculation.py' on the ../../scripts/ directory and write a function in it to calculate pi to the 5th digit.

    Then, create a 'pi_calculation_test.py' file on the same directory to test your implementation.
    """,
)

run_conversation(messages)

I'll help you create both files. Let me start by creating the `pi_calculation.py` file with a function to calculate pi to the 5th digit, and then create a test file for it.
Now let me create the test file:
Perfect! I've created both files for you. Here's a summary of what I've created:

## 1. `pi_calculation.py`
This file contains multiple methods for calculating π to at least 5 decimal places (3.14159):

- **`calculate_pi_leibniz()`**: Uses the Leibniz formula (π/4 = 1 - 1/3 + 1/5 - 1/7 + ...)
- **`calculate_pi_nilakantha()`**: Uses the Nilakantha series (faster convergence than Leibniz)
- **`calculate_pi_chudnovsky()`**: Uses the Chudnovsky algorithm (one of the fastest methods)
- **`calculate_pi_monte_carlo()`**: Uses Monte Carlo random sampling method
- **`calculate_pi()`**: A wrapper function that allows you to choose any method

All methods are tuned with default parameters that ensure accuracy to at least 5 decimal places.

## 2. `pi_calculation_test.py`
This comprehensive test 

[{'role': 'user',
  'content': "\n    Create a new file called 'pi_calculation.py' on the ../../scripts/ directory and write a function in it to calculate pi to the 5th digit.\n\n    Then, create a 'pi_calculation_test.py' file on the same directory to test your implementation.\n    "},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you create both files. Let me start by creating the `pi_calculation.py` file with a function to calculate pi to the 5th digit, and then create a test file for it.", type='text'),
   ToolUseBlock(id='toolu_013PGoTqhrNULmVBfaL94n3r', caller=DirectCaller(type='direct'), input={'command': 'create', 'path': '../../scripts/pi_calculation.py', 'file_text': '"""\nModule for calculating the value of pi using various methods.\n"""\n\nimport math\n\n\ndef calculate_pi_leibniz(iterations=1000000):\n    """\n    Calculate pi using the Leibniz formula.\n    \n    The Leibniz formula: π/4 = 1 - 1/3 + 1/5 - 1/7 + 1/9 - ...\n    \n    Args:\n